# Survey Sentiment Scoring

Extracts written survey responses, filters for new comments, and scores sentiment to feed
the satisfaction analytics pipeline.

Stack: PySpark, Microsoft Fabric lakehouse.

Note: internal identifiers are anonymized, and outputs are cleared because the originals
ran on internal data.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pyspark.sql.functions import col
import json

# Load Data

In [ ]:
events_survey_spark= spark.sql("SELECT event_id, created_at, audited_changes \
                                FROM Bronze_Lakehouse.rp.Events \
                                WHERE event_type = 'MaintenanceTicket' \
                                AND classification = 'tenant_survey'")
events_survey_spark = events_survey_spark.select(['event_id', 'created_at', 'audited_changes'])

# Extracting the Survey Response

In [ ]:
## Extracting the survey responses

from pyspark.sql import functions as F
from pyspark.sql.types import StringType, IntegerType, BooleanType

# Sorting data
events_survey = events_survey_spark.orderBy(col("event_id"), col("created_at").desc())

# Drop duplicates and keep the last occurrence
events_survey = events_survey.dropDuplicates(["event_id"])


# Define UDFs to extract survey data
def extract_comment(dict_str):
    try:
        dict_obj = json.loads(dict_str)
        return dict_obj.get('tenant_comments', None) if isinstance(dict_obj, dict) else None
    except (ValueError, TypeError):
        return None

def extract_recommend(dict_str):
    try:
        dict_obj = json.loads(dict_str)
        return dict_obj.get('recommendation_score', None) if isinstance(dict_obj, dict) else None
    except (ValueError, TypeError):
        return None

def extract_happy(dict_str):
    try:
        dict_obj = json.loads(dict_str)
        return dict_obj.get('satisfied_with_work_done', None) if isinstance(dict_obj, dict) else None
    except (ValueError, TypeError):
        return None


# Register the UDFs
extract_comment_udf = F.udf(extract_comment, StringType())
extract_recommend_udf = F.udf(extract_recommend, StringType())
extract_happy_udf = F.udf(extract_happy, StringType())

# Apply the UDFs to extract survey data
events_survey = events_survey.withColumn("comments", extract_comment_udf(F.col("audited_changes")))
events_survey = events_survey.withColumn("recommendation", extract_recommend_udf(F.col("audited_changes")))
events_survey = events_survey.withColumn("happy", extract_happy_udf(F.col("audited_changes")))

# change dtype of recommendation column to integer
events_survey = events_survey.withColumn("recommendation", events_survey["recommendation"].cast(IntegerType()))

# change dtype of happy column to boolean
events_survey = events_survey.withColumn("happy", events_survey["happy"].cast(BooleanType()))

# Rename events_survey column created_at to 'survey_created_at'
events_survey = events_survey.withColumnRenamed("created_at", "survey_date")

# Drop useless column
events_survey = events_survey.drop('audited_changes')

## Filtering For New Comments

In [ ]:
# Load the existing data from the lakehouse
existing_data = spark.sql("SELECT * FROM Gold_AnalyticalData.Gold_Lakehouse.rp.survey_sentiment_score")

# Identify new comments by comparing with existing data
new_comments = events_survey.join(existing_data, on=["event_id" , 'survey_date'], how="left_anti").filter(col("comments").isNotNull())

# Sentiment Analysis

In [ ]:
from pyspark.sql.functions import udf, col
from pyspark.sql.types import StringType, FloatType, StructType, StructField
from transformers import pipeline, AutoTokenizer

model_id = "cardiffnlp/twitter-roberta-base-sentiment-latest"

# Load the tokenizer for the specified model
tokenizer = AutoTokenizer.from_pretrained(model_id, force_download=True)

# Load the sentiment analysis pipeline
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model=model_id,
    top_k=None,  # Return all scores, not just the top 1 result
)

# Define a function to clean the text
def clean_text(text):
    """Remove leading and trailing backticks and ensure text is a string."""
    return str(text).strip('`') if text else ""

# Function to convert sentiment scores to a float between -1 and 1
def get_sentiment_label_and_score(text):
    try:
        # Ensure text is a string and truncate if necessary
        text = text[:1000000]  # Limit to first million characters
        
        # Truncate the input text to the maximum length the model can handle
        max_length = min(tokenizer.model_max_length, 512)  # Use 512 as a safe maximum
        encoded_input = tokenizer(text, truncation=True, max_length=max_length, return_tensors="pt")
        truncated_text = tokenizer.decode(encoded_input["input_ids"][0], skip_special_tokens=True)
        
        results = sentiment_pipeline(truncated_text)[0]

        # Extract scores
        negative_score = next(item['score'] for item in results if item['label'] == 'negative')
        neutral_score = next(item['score'] for item in results if item['label'] == 'neutral')
        positive_score = next(item['score'] for item in results if item['label'] == 'positive')
        
        # Calculate score: -1 to 1
        score = (positive_score - negative_score) 

        # Determine the label with the highest score
        label = max(results, key=lambda x: x['score'])['label']

        return label, score

    except Exception as e:
        print(f"Error processing text: {str(e)[:200]}...")  # Print first 100 chars of error
        return "error", 0.0  # Return a default value in case of error

# Register the UDFs
clean_text_udf = udf(clean_text, StringType())
get_sentiment_label_and_score_udf = udf(
    lambda text: get_sentiment_label_and_score(text),
    StructType([
        StructField("Sentiment_Label", StringType(), True),
        StructField("Sentiment_Score", FloatType(), True)
    ])
)

# only comments
df = new_comments.filter(col("comments").isNotNull()).select('event_id', 'survey_date', 'comments')

# Clean the comments
df = df.withColumn("comments", clean_text_udf(col("comments")))

# Apply sentiment analysis to each comment
df = df.withColumn("Sentiment_Analysis", get_sentiment_label_and_score_udf(col("comments")))

# Split the sentiment analysis results into separate columns
df = df.withColumn("Sentiment_Label", col("Sentiment_Analysis.Sentiment_Label"))
df = df.withColumn("Sentiment_Score", col("Sentiment_Analysis.Sentiment_Score"))

# Join data
new_comments = new_comments.join(df.select('event_id', 'Sentiment_Label', 'Sentiment_Score'), on='event_id', how='left')

In [ ]:
from pyspark.sql.functions import current_date

# Save the DataFrame as a table in the specified schema (append mode)
new_comments.write.mode("append").format("delta").save("abfss://Gold_AnalyticalData@onelake.dfs.fabric.microsoft.com/Gold_Lakehouse.Lakehouse/Tables/rp/survey_sentiment_score")